# Poker Aid Final Project: Mega Notebook

This notebook is the full narrative report. The longer helper code lives in `mega_final_project_helpers.py` in this same `notebooks/` folder and is imported in the setup cell below.

We move in chronological order from question -> data -> preprocessing -> analysis -> modeling -> interpretation.

The goal is practical: estimate
- the player's **win chance** by stage (preflop/flop/turn/river), and
- opponents' **bluff probability** from visible behavior.

---

## Roadmap
1. Research Question
2. Hypothesis
3. Dataset
4. Data preprocessing
5. Data analysis and visualization
6. Data modeling and prediction
7. Results analysis, bias, and lessons learned

**Implementation note:** Section **6c** ties the notebook’s trained stage models to the **PyScript** poker UI: the UI’s default win % uses `poker_page/hand_eval.py` + `poker_page/heuristics.py` (hand-strength–aware proxy); the RF path uses the same hand-ranking idea as `mega_final_project_helpers.build_stage_feature_payload` / `scripts/features/poker_hand_strength.py` when `ml_enabled` and artifacts are available.

## 1) Research Question

### What we are doing
We define the exact decision problems this poker aid is trying to solve.

### Why it matters
A useful poker model should not just be accurate in abstract; it should support real in-game choices.

### Core questions
1. Can we estimate a player's **win probability** during each stage of the hand using cards, board, stack, pot, and position context?
2. Can we estimate an opponent's **bluff probability** using only **visible** information (betting behavior, board texture, stack/pot pressure), without hole cards?
3. Are the modeled relationships statistically meaningful, and can we communicate uncertainty clearly?

## 2) Hypothesis

### Primary hypotheses
- **H1 (Win Chance):** Stage-specific features (hand strength proxies, board texture, stack pressure, and position) predict `won_flag` better than chance baseline.
- **H2 (Bluff):** Visible behavior/context features predict `is_bluffing` better than chance baseline.

### Statistical form
- **H0:** Predictor set has no association with target beyond random variation.
- **Ha:** Predictor set has meaningful association with target.

### Evidence we will use
- Predictive metrics (AUC, Brier, accuracy/F1)
- Hypothesis tests (Welch t-test, Kruskal-Wallis, chi-square)
- Effect direction and practical interpretation

## 3) Dataset

### What we are doing
Load and document the source dataset and derived targets/features.

### Dataset summary
- **Main source:** `data/gambling.csv`
- **Project-aligned cleaned dataset (created in this notebook):** `data/cleanedGambling.csv`

### Why this dataset exists
The hand-history data captures player actions, board cards, stacks, and outcomes from online poker sessions. These fields allow behavioral and stage-based modeling.

### Key variables (high-level)
- **Identifiers/context:** `hand_id`, `tourn_id`, `table`, `seat`, `position`, `date`, `time`
- **Cards/board:** `cards`, `board_flop`, `board_turn`, `board_river`
- **Betting/pot:** `bet_pre`, `bet_flop`, `bet_turn`, `bet_river`, `pot_pre`, `pot_flop`, `pot_turn`, `pot_river`
- **Outcome:** `result`, `balance`
- **Derived targets:** `won_flag`, `is_bluffing`

### Reference note
This notebook integrates methods consistent with course labs (EDA, hypothesis testing, dimensionality reduction, regression/classification, bootstrap uncertainty).

In [ ]:
# Core imports
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

# Ensure we can import `mega_final_project_helpers.py` whether cwd is repo root or `notebooks/`
_here = Path.cwd().resolve()
_candidates = [_here, _here / "notebooks"]
for d in _candidates:
    if (d / "mega_final_project_helpers.py").exists():
        p = str(d)
        if p not in sys.path:
            sys.path.insert(0, p)
        break

from mega_final_project_helpers import (
    CLEAN_PATH,
    RAW_PATH,
    ROOT,
    RANDOM_SEED,
    STAGE_FEATURES,
    VISIBLE_BLUFF_FEATURES,
    build_cleaned_gambling_dataframe,
    build_stage_training_frame,
    build_visible_bluff_frame,
    evaluate_classifier_with_group_split,
    stage_from_board_row,
    visible_vector_from_csv_row,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

np.random.seed(RANDOM_SEED)

print("Project root:", ROOT)
print("Raw dataset path:", RAW_PATH, "exists:", RAW_PATH.exists())


ModuleNotFoundError: No module named 'seaborn'

In [ ]:
# Project helpers live in `mega_final_project_helpers.py` (same folder as this notebook)
# The next cells assume the imports from the cell above have been executed.


In [ ]:
# Load raw dataset and inspect
if not RAW_PATH.exists():
    raise FileNotFoundError(f"Could not find {RAW_PATH}. Run this notebook from project root.")

raw_df = pd.read_csv(RAW_PATH)
print("Raw shape:", raw_df.shape)
raw_df.head(3)

In [ ]:
# Quick schema + missingness preview
raw_info = pd.DataFrame({
    "column": raw_df.columns,
    "dtype": [str(raw_df[c].dtype) for c in raw_df.columns],
    "missing": [int(raw_df[c].isna().sum()) for c in raw_df.columns],
    "missing_pct": [100.0 * raw_df[c].isna().mean() for c in raw_df.columns],
}).sort_values("missing_pct", ascending=False)

display(raw_info.head(20))

fig, ax = plt.subplots(figsize=(12, 4))
raw_info.sort_values("missing_pct", ascending=False).head(20).plot(
    x="column", y="missing_pct", kind="bar", ax=ax, legend=False, color="#4C72B0"
)
ax.set_title("Top 20 Columns by Missing % (Raw)")
ax.set_ylabel("Missing %")
ax.set_xlabel("Column")
ax.tick_params(axis="x", rotation=70)
plt.tight_layout()
plt.show()

### Interpretation (dataset)

We can already see that the dataset has mixed data types and missingness concentrated in some stage-specific fields, which is expected in hand-history data. This supports using a documented preprocessing step before any modeling.

Next we create a cleaned dataset with reproducible target/feature engineering and save it to `data/cleanedGambling.csv`.

## 4) Data Preprocessing

### What we are doing
Build the cleaned dataset using the functions in `mega_final_project_helpers.py` (imported above), executed from this notebook.

### Why it matters
Our downstream statistical tests and models are only valid if typing, missing values, and target construction are handled consistently.

### Decisions
- Keep rows unless core keys are unusable.
- Coerce numeric columns safely and fill model-time NaNs with 0.0 where appropriate.
- Use deterministic hashed `player_id` to remove direct names.
- Build targets and behavior features (`won_flag`, `is_bluffing`, streaks, equity proxy).

In [ ]:
# Build and save cleaned dataset
clean_df = build_cleaned_gambling_dataframe(raw_df)
clean_df.to_csv(CLEAN_PATH, index=False)

print("Clean shape:", clean_df.shape)
print("Saved:", CLEAN_PATH)

# Missingness report before/after for shared columns
shared_cols = [c for c in raw_df.columns if c in clean_df.columns]
missing_compare = pd.DataFrame({
    "column": shared_cols,
    "raw_missing_pct": [100.0 * raw_df[c].isna().mean() for c in shared_cols],
    "clean_missing_pct": [100.0 * clean_df[c].isna().mean() for c in shared_cols],
})
missing_compare["delta_pct"] = missing_compare["clean_missing_pct"] - missing_compare["raw_missing_pct"]
display(missing_compare.sort_values("raw_missing_pct", ascending=False).head(20))

clean_df.head(3)

In [ ]:
# Data quality checks used to justify preprocessing decisions
quality = pd.DataFrame({
    "metric": [
        "rows", "unique_hands", "unique_players",
        "won_flag_rate", "bluff_rate", "all_in_rate"
    ],
    "value": [
        len(clean_df),
        clean_df.get("hand_id", pd.Series(dtype=str)).astype(str).nunique(),
        clean_df.get("player_id", pd.Series(dtype=str)).astype(str).nunique(),
        float(clean_df.get("won_flag", pd.Series([0])).mean()),
        float(clean_df.get("is_bluffing", pd.Series([0])).mean()),
        float(clean_df.get("is_all_in", pd.Series([False])).astype(float).mean()),
    ],
})
display(quality)

num_cols = [c for c in ["stack", "bet_pre", "bet_flop", "bet_turn", "bet_river", "pot_pre", "pot_flop", "pot_turn", "pot_river", "net_result"] if c in clean_df.columns]
if num_cols:
    display(clean_df[num_cols].describe().T)

### Interpretation (preprocessing)

The cleaned dataset preserves raw hand context while adding analysis-ready targets and behavior features. Missing values are now explicit and controlled, and we can proceed to exploratory analysis and formal tests with a stable schema.

## 5) Data Analysis and Visualization

### What we are doing
Use descriptive statistics and visual summaries to understand distributions and relationships before fitting models.

### Why it matters
EDA helps identify informative variables, class imbalance, outliers, and plausible mechanisms behind win/bluff behavior.

In [ ]:
# Central tendency + distributions
eda_cols = [c for c in ["starting_stack", "aggression_score", "preflop_equity", "strength_mean", "net_result", "win_streak"] if c in clean_df.columns]

if eda_cols:
    display(clean_df[eda_cols].describe().T)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()
for i, col in enumerate(eda_cols[:6]):
    sns.histplot(clean_df[col].dropna(), kde=True, ax=axes[i], color="#1f77b4")
    axes[i].set_title(f"Distribution: {col}")
for j in range(len(eda_cols[:6]), 6):
    axes[j].axis("off")
plt.tight_layout()
plt.show()

if {"starting_stack", "net_result"}.issubset(clean_df.columns):
    plt.figure(figsize=(7, 4))
    sns.scatterplot(data=clean_df.sample(min(len(clean_df), 4000), random_state=RANDOM_SEED), x="starting_stack", y="net_result", alpha=0.4)
    plt.title("Starting Stack vs Net Result")
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation structure
corr_cols = [c for c in [
    "won_flag", "is_bluffing", "aggression_score", "strength_mean", "preflop_equity",
    "starting_stack", "table_position", "win_streak", "net_result"
] if c in clean_df.columns]

corr_df = clean_df[corr_cols].copy()
for c in corr_df.columns:
    corr_df[c] = pd.to_numeric(corr_df[c], errors="coerce")

corr = corr_df.corr(numeric_only=True)
plt.figure(figsize=(9, 6))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

# Simple grouped view
if {"is_bluffing", "net_result"}.issubset(clean_df.columns):
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=clean_df, x="is_bluffing", y="net_result")
    plt.title("Net Result by Bluff Label")
    plt.tight_layout()
    plt.show()

### Interpretation (EDA)

The distributions and correlations provide a first pass on which features may be informative. In particular, aggression, strength proxies, and stack-related variables tend to show meaningful structure. We now move from descriptive evidence to formal statistical testing.

## 5b) Hypothesis Testing

### What we are doing
Run class-aligned tests to evaluate relationships quantitatively.

### Tests
- Welch two-sample t-test
- Kruskal-Wallis (non-parametric)
- Chi-square test of independence

In [ ]:
# Statistical tests
results_rows = []

# 1) Welch t-test: aggression by won_flag
if {"aggression_score", "won_flag"}.issubset(clean_df.columns):
    g1 = clean_df.loc[clean_df["won_flag"] == 1, "aggression_score"].dropna()
    g0 = clean_df.loc[clean_df["won_flag"] == 0, "aggression_score"].dropna()
    if len(g1) > 2 and len(g0) > 2:
        t_stat, p_val = stats.ttest_ind(g1, g0, equal_var=False)
        results_rows.append({
            "test": "Welch t-test (aggression | won_flag)",
            "statistic": float(t_stat),
            "p_value": float(p_val),
            "significant_0_05": bool(p_val < 0.05),
        })

# 2) Kruskal-Wallis: net_result across win_streak bins
if {"net_result", "win_streak"}.issubset(clean_df.columns):
    tmp = clean_df.copy()
    tmp["streak_bin"] = pd.cut(tmp["win_streak"], bins=[-1, 0, 2, 5, 100], labels=["0", "1-2", "3-5", "6+"])
    groups = [g["net_result"].dropna().values for _, g in tmp.groupby("streak_bin", observed=False)]
    groups = [g for g in groups if len(g) > 2]
    if len(groups) >= 2:
        h_stat, p_val = stats.kruskal(*groups)
        results_rows.append({
            "test": "Kruskal-Wallis (net_result by streak bin)",
            "statistic": float(h_stat),
            "p_value": float(p_val),
            "significant_0_05": bool(p_val < 0.05),
        })

# 3) Chi-square: all-in vs won_flag
if {"is_all_in", "won_flag"}.issubset(clean_df.columns):
    ct = pd.crosstab(clean_df["is_all_in"].astype(int), clean_df["won_flag"].astype(int))
    if ct.shape == (2, 2):
        chi2, p_val, dof, expected = stats.chi2_contingency(ct)
        results_rows.append({
            "test": "Chi-square (is_all_in vs won_flag)",
            "statistic": float(chi2),
            "p_value": float(p_val),
            "significant_0_05": bool(p_val < 0.05),
        })

stats_results = pd.DataFrame(results_rows)
display(stats_results if not stats_results.empty else pd.DataFrame({"info": ["Not enough data for selected tests"]}))

### Interpretation (hypothesis testing)

We use p-values as one form of evidence, but we do not treat statistical significance as the only decision criterion. The practical direction of effects and model performance under holdout evaluation are equally important for deciding whether these features are genuinely useful in a poker aid.

## 5c) Dimensionality Reduction and Clustering

### What we are doing
Use PCA + KMeans on behavioral features to explore player archetypes.

### Why it matters
This can reveal latent behavior patterns that are not obvious from one variable at a time.

In [ ]:
cluster_cols = [c for c in [
    "aggression_score", "strength_mean", "preflop_equity", "table_position",
    "starting_stack", "win_streak", "loss_streak"
] if c in clean_df.columns]

cluster_view = clean_df[cluster_cols].copy().apply(pd.to_numeric, errors="coerce").fillna(0.0)

if len(cluster_view) >= 50 and len(cluster_cols) >= 3:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(cluster_view)

    pca = PCA(n_components=2, random_state=RANDOM_SEED)
    X_pca = pca.fit_transform(X_scaled)

    kmeans = KMeans(n_clusters=4, random_state=RANDOM_SEED, n_init=20)
    clusters = kmeans.fit_predict(X_scaled)

    pca_df = pd.DataFrame({"pc1": X_pca[:, 0], "pc2": X_pca[:, 1], "cluster": clusters})

    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=pca_df.sample(min(len(pca_df), 5000), random_state=RANDOM_SEED), x="pc1", y="pc2", hue="cluster", alpha=0.6, palette="tab10")
    plt.title("PCA + KMeans behavior clusters")
    plt.tight_layout()
    plt.show()

    print("Explained variance ratio:", pca.explained_variance_ratio_)
    cluster_profile = pd.concat([cluster_view, pd.Series(clusters, name="cluster")], axis=1).groupby("cluster").mean(numeric_only=True)
    display(cluster_profile)
else:
    print("Not enough data/columns for PCA + clustering block.")

### Interpretation (dimensionality reduction)

The PCA projection gives an interpretable low-dimensional view of player behavior, while clustering groups similar strategy profiles. These profiles can support future personalization (for example, opponent-specific bluff calibration).

## 6) Data Modeling and Prediction

### What we are doing
Train and evaluate two classification models:
1. Stage-based win chance (`won_flag`)
2. Visible-feature bluff probability (`is_bluffing`)

### Important methodology choice
We split by `hand_id` using `GroupShuffleSplit` to reduce leakage between train and test.

The grouped train/test evaluation helper `evaluate_classifier_with_group_split` now lives in `mega_final_project_helpers.py` (imported in the core imports cell).


In [ ]:
# Train stage win models (preflop/flop/turn/river)
stage_models = {}
stage_metrics = []
stage_importances = {}

for stage in ["preflop", "flop", "turn", "river"]:
    stage_frame = build_stage_training_frame(clean_df, stage)
    needed = [c for c in STAGE_FEATURES[stage] if c in stage_frame.columns]
    if len(stage_frame) < 100 or len(needed) < 5:
        print(f"Skipping {stage}: insufficient rows or features.")
        continue
    model, m, imp = evaluate_classifier_with_group_split(
        stage_frame,
        features=needed,
        target="won_flag",
        group_col="hand_id",
        label=f"stage_win_{stage}",
    )
    stage_models[stage] = model
    stage_metrics.append(m)
    stage_importances[stage] = imp

stage_metrics_df = pd.DataFrame(stage_metrics).sort_values("auc", ascending=False)
display(stage_metrics_df if not stage_metrics_df.empty else pd.DataFrame({"info": ["No stage models trained"]}))

for stage, imp in stage_importances.items():
    print(f"\nTop features for {stage}")
    display(imp.head(8))

In [ ]:
# Train visible bluff model
bluff_frame = build_visible_bluff_frame(clean_df)
bluff_features = [c for c in VISIBLE_BLUFF_FEATURES if c in bluff_frame.columns]

bluff_model = None
bluff_metrics_df = pd.DataFrame()
bluff_importance = pd.DataFrame()

if len(bluff_frame) >= 100 and len(bluff_features) >= 5:
    bluff_model, bluff_metrics, bluff_importance = evaluate_classifier_with_group_split(
        bluff_frame,
        features=bluff_features,
        target="is_bluffing",
        group_col="hand_id",
        label="visible_bluff",
    )
    bluff_metrics_df = pd.DataFrame([bluff_metrics])
    display(bluff_metrics_df)
    display(bluff_importance.head(12))
else:
    print("Skipping visible bluff model: insufficient rows/features")

### Regression add-on (class coverage)

We include a compact regression analysis to model `net_result` from selected numeric features, then bootstrap coefficient stability.

In [ ]:
# Linear regression + bootstrap coefficient stability
reg_features = [c for c in [
    "aggression_score", "preflop_equity", "strength_mean", "table_position", "starting_stack", "win_streak"
] if c in clean_df.columns]

if reg_features and "net_result" in clean_df.columns:
    reg_df = clean_df[reg_features + ["net_result"]].copy()
    reg_df = reg_df.apply(pd.to_numeric, errors="coerce").fillna(0.0)

    X = reg_df[reg_features]
    y = reg_df["net_result"]

    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=reg_features)

    split = int(0.8 * len(X_scaled))
    X_train, X_test = X_scaled.iloc[:split], X_scaled.iloc[split:]
    y_train, y_test = y.iloc[:split], y.iloc[split:]

    lr = LinearRegression()
    lr.fit(X_train, y_train)
    y_pred = lr.predict(X_test)

    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    print("Linear regression RMSE:", round(float(rmse), 4))

    coef_df = pd.DataFrame({"feature": reg_features, "coef": lr.coef_}).sort_values("coef", key=np.abs, ascending=False)
    display(coef_df)

    # Residual view
    residuals = y_test - y_pred
    plt.figure(figsize=(6, 4))
    plt.scatter(y_pred, residuals, alpha=0.3)
    plt.axhline(0, color="red", linestyle="--")
    plt.title("Residual Plot (Regression)")
    plt.xlabel("Predicted net_result")
    plt.ylabel("Residual")
    plt.tight_layout()
    plt.show()

    # Bootstrap coefficient signs
    B = 200
    coef_rows = []
    n = len(X_scaled)
    for _ in range(B):
        idx = np.random.choice(n, size=n, replace=True)
        xb = X_scaled.iloc[idx]
        yb = y.iloc[idx]
        m = LinearRegression().fit(xb, yb)
        coef_rows.append(m.coef_)
    boot = pd.DataFrame(coef_rows, columns=reg_features)
    boot_summary = pd.DataFrame({
        "feature": reg_features,
        "coef_mean": boot.mean().values,
        "coef_std": boot.std().values,
        "pct_positive": (boot > 0).mean().values * 100,
    }).sort_values("coef_std", ascending=False)
    display(boot_summary)
else:
    print("Skipping regression block: required columns not available.")

### Interpretation (modeling)

If stage and bluff models perform above baseline (especially by AUC and Brier), this supports the main hypotheses. Feature importance and bootstrap stability help us explain *why* the model behaves as it does, instead of presenting black-box predictions only.

## 6b) Independent Inference Demo

### What we are doing
Use one sampled hand row to produce both predictions:
- stage win chance
- visible bluff probability

In [ ]:
# Demo inference from one random row
if len(clean_df) == 0:
    print("No rows available for demo")
else:
    demo_row = clean_df.sample(1, random_state=RANDOM_SEED).iloc[0]
    stage = stage_from_board_row(demo_row)

    if stage in stage_models:
        stage_frame = build_stage_training_frame(pd.DataFrame([demo_row]), stage)
        fcols = [c for c in STAGE_FEATURES[stage] if c in stage_frame.columns]
        win_prob = float(stage_models[stage].predict_proba(stage_frame[fcols].fillna(0.0))[0, 1])
    else:
        win_prob = np.nan

    if bluff_model is not None:
        bluff_vec = visible_vector_from_csv_row(demo_row)
        xb = pd.DataFrame([bluff_vec]).reindex(columns=bluff_features).fillna(0.0)
        bluff_prob = float(bluff_model.predict_proba(xb)[0, 1])
    else:
        bluff_prob = np.nan

    print("Demo hand_id:", demo_row.get("hand_id", "unknown"))
    print("Detected stage:", stage)
    print("Predicted win chance:", f"{win_prob:.2%}" if np.isfinite(win_prob) else "N/A")
    print("Predicted bluff probability:", f"{bluff_prob:.2%}" if np.isfinite(bluff_prob) else "N/A")

    if np.isfinite(win_prob) and np.isfinite(bluff_prob):
        if win_prob > 0.60 and bluff_prob > 0.55:
            msg = "Model suggests strong hand context against potentially aggressive/bluffy line."
        elif win_prob < 0.40 and bluff_prob < 0.35:
            msg = "Model suggests caution: weaker context and lower visible bluff signal."
        else:
            msg = "Mixed signal: consider table dynamics and risk tolerance."
        print("Decision-style summary:", msg)

## 6c) Trained model vs live PyScript UI

### Two win-probability paths in the repo

1. **Notebook / training / optional `ml_enabled` server path**  
   - **Features:** `mega_final_project_helpers.build_stage_feature_payload` mirrors the contract of [`scripts/features/poker_hand_strength.py`](../../scripts/features/poker_hand_strength.py) (`hand_strength` = numeric tier from `best_combination_from_tokens`, board texture, draws, stacks, SPR, etc.).  
   - **Predictor:** `RandomForestClassifier` (+ calibration) per street, trained from `cleanedGambling.csv`; inference via `predict_proba` on the stage feature vector (see section 6b).

2. **Default poker UI (PyScript in [`poker_page/`](../../poker_page/))**  
   - **`GameState.ml_enabled`** is `False` by default, and the browser bundle does not ship `scripts/` or `poker_models.pkl`, so the **sklearn stage model is not used** in the usual static page.  
   - **Instead:** [`poker_page/hand_eval.py`](../../poker_page/hand_eval.py) evaluates the **best 7-card combination** (same category ordering as training: high card → straight flush).  
   - [`poker_page/heuristics.py`](../../poker_page/heuristics.py) **`hero_win_probability_proxy`** maps that tier (plus street, villain count, board texture) to a **[0, 1]** display value. It is a **transparent proxy**, not the trained RF.

### Takeaway for the report

The notebook models remain the authoritative **data-driven** win predictors. The live UI proxy is **aligned on hand classification** with the training feature builder but uses a **simpler closed-form** multway adjustment so the page stays fast and dependency-light in WASM.


In [ ]:
# 6c — Compare notebook hand label vs. poker_page (same logic family as training `hand_strength`)
import sys
from pathlib import Path

_pp = ROOT / "poker_page"
if _pp.is_dir() and str(_pp) not in sys.path:
    sys.path.insert(0, str(_pp))

from hand_eval import best_combination_from_tokens as ui_best
from hand_eval import hand_strength_index as ui_idx
from hand_eval import tuples_to_tokens
from heuristics import hero_win_probability_proxy

from mega_final_project_helpers import best_combination_from_tokens as nb_best
from mega_final_project_helpers import hand_strength_from_tokens as nb_hand_strength

hole = [("A", "S"), ("A", "H")]
board = [("K", "D"), ("K", "C"), ("K", "H"), ("Q", "S"), ("2", "C")]
toks = tuples_to_tokens(hole + board)

print("Compact tokens:", toks)
print("poker_page best:", ui_best(toks), "| COMBINATION index", ui_idx(toks))
print("notebook helper best:", nb_best(toks), "| hand_strength feature", nb_hand_strength(toks))
print("Live UI win proxy vs 5 villains:", f"{hero_win_probability_proxy(hole, board, 5):.1%}")
print("Live UI win proxy vs 1 villain:", f"{hero_win_probability_proxy(hole, board, 1):.1%}")


## 7) Results Analysis

### Hypothesis check
- We assess support for hypotheses using both statistical tests and holdout model metrics.
- Stronger evidence comes from agreement across multiple signals (AUC/Brier + test direction + feature stability).

### Statistical significance vs practical significance
A statistically significant effect can still be too small to matter in real play. We focus on whether outputs are stable and actionable, not only whether p < 0.05.

### Bias and limitations
- Labels like `is_bluffing` are heuristics, not ground-truth psychology.
- Hand-history populations may not represent all player pools/stakes.
- Deterministic proxy features (like synthetic strength/equity proxies) add approximation error.
- The **live PyScript win %** is a **closed-form proxy** (tier + villains + board texture), not Monte Carlo equity and not the trained RF unless ML artifacts are enabled in the client.
- Temporal and opponent adaptation effects are only partially captured.

### Potential negative implications
- Over-reliance on model output can reduce human strategic flexibility.
- Models may amplify historical biases from specific pools/opponents.
- Misuse in competitive settings may raise fairness or platform-policy concerns.

### Improvements
- Add richer externally validated labels and stronger hand-evaluation ground truth.
- Perform opponent-segment calibration and temporal drift checks.
- Add uncertainty intervals directly on live predictions.
- Run repeated grouped CV and threshold optimization for deployment.

## Conclusion

This notebook provides a full end-to-end workflow for the poker aid problem:
- from raw hand-history data,
- through cleaning, EDA, testing, dimensionality reduction, and model training,
- to interpretable win/bluff predictions.

Implementation details are centralized in `mega_final_project_helpers.py` so the notebook stays readable while remaining reproducible.

**Live app alignment:** The PyScript table at [`poker_page/`](../poker_page/) shows win % from [`hand_eval.py`](../poker_page/hand_eval.py) (7-card best-hand categories) plus [`heuristics.py`](../poker_page/heuristics.py) `hero_win_probability_proxy` when ML artifacts are not loaded in the browser. That proxy uses the **same combination ordering** as the `hand_strength` feature in this notebook and in [`scripts/features/poker_hand_strength.py`](../scripts/features/poker_hand_strength.py), but it is **not** the calibrated RF unless `ml_enabled` and bundled models are wired in.

The project shows that behavior and context features can provide useful predictive signal, while also highlighting important limits, uncertainty, and responsible-use considerations.